# 01 · Exploratory Data Analysis

**Question:** Is the Superstore export clean enough to trust, and what does it contain?

**Answer:** Yes, once dates are parsed day-first. It holds 9,800 order lines (4,922 orders, 793 customers) from Jan 2015 to Dec 2018, $2.26M in sales. The only gaps are 11 missing postal codes, and every shipping delay falls between 0 and 7 days.

## Dataset
- Source: public Superstore sample dataset (`data/raw/superstore.csv`)
- Data type: transaction-level retail order lines
- Key fields: Order Date, Ship Date, Ship Mode, Segment, Region, State, Category, Sub-Category, Sales
- Not included: Profit, Quantity, Discount, so this project makes no margin claims

## What this notebook checks
- Shape, data types and missing values
- Date parsing (the source dates are DD/MM/YYYY)
- The shipping delay KPI: days from order to shipment, by ship mode

## Tools
- Python (pandas, matplotlib, seaborn)
- Tidy CSV outputs in `data/processed/`, ready for Excel pivot tables
- Tableau Public for the dashboard

In [1]:
import sys
print("Python executable:", sys.executable)
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option("display.max_columns", None)

# Load dataset
df = pd.read_csv("../data/raw/superstore.csv")

# Preview data
df.head()

Python executable: /tmp/claude-0/-home-user/2fff10fd-ce4c-5635-ae85-d28d46663624/scratchpad/venv/bin/python3


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680


In [2]:
import pandas as pd

df = pd.read_csv("../data/raw/superstore.csv")

df.shape
df.head()
df.info()
df.describe()



<class 'pandas.DataFrame'>
RangeIndex: 9800 entries, 0 to 9799
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9800 non-null   int64  
 1   Order ID       9800 non-null   str    
 2   Order Date     9800 non-null   str    
 3   Ship Date      9800 non-null   str    
 4   Ship Mode      9800 non-null   str    
 5   Customer ID    9800 non-null   str    
 6   Customer Name  9800 non-null   str    
 7   Segment        9800 non-null   str    
 8   Country        9800 non-null   str    
 9   City           9800 non-null   str    
 10  State          9800 non-null   str    
 11  Postal Code    9789 non-null   float64
 12  Region         9800 non-null   str    
 13  Product ID     9800 non-null   str    
 14  Category       9800 non-null   str    
 15  Sub-Category   9800 non-null   str    
 16  Product Name   9800 non-null   str    
 17  Sales          9800 non-null   float64
dtypes: float64(2), int6

,Row ID,Postal Code,Sales
count,9800.000000,9789.000000,9800.000000
mean,4900.500000,55273.322403,230.769059
std,2829.160653,32041.223413,626.651875
min,1.000000,1040.000000,0.444000
25%,2450.750000,23223.000000,17.248000
50%,4900.500000,58103.000000,54.490000
75%,7350.250000,90008.000000,210.605000
max,9800.000000,99301.000000,22638.480000


## Initial Observations

- The dataset contains 9,800 order lines, which roll up to 4,922 unique orders.
- Sales values are highly skewed: the median line is $54.49, the mean is $230.77, and the largest single line is $22,638.48.
- 11 records are missing a postal code; every other field is complete.

In [3]:
df[['Order Date', 'Ship Date']].head()
df[['Order Date', 'Ship Date']].dtypes


Order Date    str
Ship Date     str
dtype: object

In [4]:
# Convert date columns to datetime (coerce invalids).
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True, errors='coerce')  # source dates are DD/MM/YYYY
df['Ship Date'] = pd.to_datetime(df['Ship Date'], dayfirst=True, errors='coerce')
# Show parsed dtypes (detailed KPI derivation follows in later cells)
df[['Order Date','Ship Date']].dtypes


Order Date    datetime64[us]
Ship Date     datetime64[us]
dtype: object

In [5]:
df[['Order Date', 'Ship Date']].dtypes


Order Date    datetime64[us]
Ship Date     datetime64[us]
dtype: object

In [6]:
# Confirm the date columns types after coercion (no strict re-parsing here)
df[['Order Date', 'Ship Date']].dtypes


Order Date    datetime64[us]
Ship Date     datetime64[us]
dtype: object

## Shipping KPI derivation and analysis

This section derives `Shipping Delay (Days)` as a core operational KPI, handles mixed/invalid dates, and computes descriptive and comparative statistics by `Ship Mode`, `Category`, and `Region`.

In [7]:
# Robust date parsing and KPI derivation
import pandas as pd
from dateutil import parser

def safe_parse_dates(series):
    # First try pandas fast path, then fallback to dateutil for remaining values
    s = pd.to_datetime(series, dayfirst=True, errors='coerce')
    mask = s.isna()
    if mask.any():
        def _try_parse(x):
            try:
                return parser.parse(str(x))
            except Exception:
                return pd.NaT
        s.loc[mask] = series[mask].apply(_try_parse)
    return pd.to_datetime(s, errors='coerce')

# Apply safe parsing to the order and ship date columns (handles mixed formats)
df['Order Date Parsed'] = safe_parse_dates(df['Order Date'])
df['Ship Date Parsed'] = safe_parse_dates(df['Ship Date'])

# Create flags for parsing issues
df['Order Date Invalid'] = df['Order Date Parsed'].isna()
df['Ship Date Invalid'] = df['Ship Date Parsed'].isna()

# Derived KPI: Shipping Delay in days (can be negative if data issue)
df['Shipping Delay (Days)'] = (df['Ship Date Parsed'] - df['Order Date Parsed']).dt.days

# Basic quality summary
quality = {
    'total_records': len(df),
    'order_date_invalid': int(df['Order Date Invalid'].sum()),
    'ship_date_invalid': int(df['Ship Date Invalid'].sum()),
    'both_dates_valid': int((~df['Order Date Invalid'] & ~df['Ship Date Invalid']).sum())
}
quality

{'total_records': 9800,
 'order_date_invalid': 0,
 'ship_date_invalid': 0,
 'both_dates_valid': 9800}

In [8]:
# Descriptive statistics for Shipping Delay
delay = df['Shipping Delay (Days)']
desc = delay.describe(percentiles=[0.25,0.5,0.75]).to_dict()
iqr = desc.get('75%') - desc.get('25%') if ('75%' in desc and '25%' in desc) else None
overall_stats = {
    'count': int(desc.get('count', 0)),
    'mean': float(desc.get('mean', float('nan'))),
    'median': float(desc.get('50%', float('nan'))),
    'std': float(desc.get('std', float('nan'))),
    'min': float(desc.get('min', float('nan'))),
    'max': float(desc.get('max', float('nan'))),
    'IQR_days': float(iqr) if iqr is not None else None
}
overall_stats

{'count': 9800,
 'mean': 3.9611224489795918,
 'median': 4.0,
 'std': 1.7496136250116285,
 'min': 0.0,
 'max': 7.0,
 'IQR_days': 2.0}

## Generated Visuals

Below are the charts generated by the shipping analysis script.

![Histogram of Shipping Delay](../visuals/shipping_delay_hist.png)

![Boxplot: Shipping Delay by Ship Mode](../visuals/shipping_delay_boxplot.png)

![Sales vs Shipping Delay](../visuals/sales_vs_delay.png)

In [9]:
# Print concise summary generated by the analysis script
print(open('../visuals/summary.txt').read())

Concise Shipping Analysis Summary
--------------------------------
Total records: 9800
Order date invalid: 0
Ship date invalid: 0

Overall shipping delay (days):
- count: 9800
- mean: 3.9611224489795918
- median: 4.0
- std: 1.7496136250116285
- min: 0.0
- max: 7.0
- iqr_days: 2.0

Ship Mode summary, per order line (median, IQR, neg_rate, long_rate_gt7d):
- Same Day: median=0.0, IQR=0.0, neg_rate=0.000, long_rate_gt7d=0.000
- First Class: median=2.0, IQR=1.0, neg_rate=0.000, long_rate_gt7d=0.000
- Second Class: median=3.0, IQR=2.0, neg_rate=0.000, long_rate_gt7d=0.000
- Standard Class: median=5.0, IQR=2.0, neg_rate=0.000, long_rate_gt7d=0.000

Suspicious records flagged: 0 (threshold > 9.2 days)

Spearman correlation (order total Sales vs Delay):
- correlation: -0.013103683908446106, p-value: 0.35803245514328874

Recommended next steps:
- No negative or extreme delays found: shipping performance is clean in this export
- Standard Class is the slowest mode; review it first if fulfillment

## Takeaway

The data is clean enough to analyze once dates are parsed day-first. Parsing them month-first (the original bug in this project) scrambles monthly totals and creates impossible delays. Next: `02_aggregate_sales_by_category.ipynb` for what drives revenue.